# Experiential Latent Memory (ELM) - Paper Reproduction Notebook

**Continual Self-Improvement with Lightweight Experiential Latent Memories** (Dorovatas et al., arXiv:2606.17803v1)

This notebook replicates all pipelines and experiments from the paper:
- **Online Continual Self-Improvement (Algorithm 1)**
- **Baselines:** Zero-shot CoT, ICL, Reasoning Bank, LAG, Verifier@2, Offline Full-Model TTRL
- **Offline Analysis:** Memory Specialization, Oracle ZS vs Mem, Flip Rates, Overlap IoU
- **Cross-Dataset Transfer:** AMC23 $\leftrightarrow$ AIME24
- **Memory Design Ablations:** Soft Prompts vs LoRA vs Prefix Tuning

## 1. Setup & GPU Verification

In [ ]:
# Check GPU
!nvidia-smi

# Optional: Mount Google Drive to persist trained memory pools and model checkpoints
from google.colab import drive
drive.mount('/content/drive')

## 2. Install Package and Dependencies (via pyproject.toml)

In [ ]:
# Install package in editable mode with all dependencies and visualization tools
!pip install -e ".[viz]" --quiet

## 3. Load Model and Benchmarks

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from elm.config import ELMConfig
from elm.data.benchmarks import load_benchmark
from elm.pipelines.online_elm import OnlineELMPipeline
from elm.pipelines.offline_elm import OfflineELMPipeline
from elm.pipelines.cross_transfer import CrossDatasetTransferPipeline
from elm.verifier.prm import PRMVerifier
from elm.evaluator import BenchmarkEvaluator, BenchmarkSummary

MODEL_ID = "Qwen/Qwen2.5-Math-7B"
print(f"Loading base model: {MODEL_ID}...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
    trust_remote_code=True
)
model.eval()
print("Model loaded successfully!")

# Load datasets
amc23_dataset = load_benchmark("AMC23")
aime24_dataset = load_benchmark("AIME24")
print(f"Loaded {len(amc23_dataset)} AMC23 items and {len(aime24_dataset)} AIME24 items.")

## 4. Run Online Continual Self-Improvement (Algorithm 1 - Table 3)

In [ ]:
config = ELMConfig()
config.retrieval.retrieval_mode = "input" # or 'kv'

online_pipeline = OnlineELMPipeline(config=config, model=model, tokenizer=tokenizer)
online_results = online_pipeline.run_stream(amc23_dataset)

print(f"\n>>> ELM Online Accuracy on AMC23: {online_results['accuracy']:.2f}%")
# Save accumulated memory pool to Google Drive
online_pipeline.memory_pool.save_to_disk("/content/drive/MyDrive/elm_amc23_memory_pool.pt")
print("Saved memory pool to Google Drive.")

## 5. Offline Memory Evaluation & Flip Analysis (Table 1 & Table 2)

In [ ]:
offline_pipe = OfflineELMPipeline(config=config, model=model, tokenizer=tokenizer)

# Train memory on sample 0
mem_sample = offline_pipe.train_memory_for_sample(amc23_dataset[0])
eval_res = offline_pipe.evaluate_memory(mem_sample, amc23_dataset, source_sample_id=amc23_dataset[0].id)

neg_rate = BenchmarkEvaluator.compute_negative_flip_rate(eval_res.negative_flips, eval_res.positive_flips)
print(f"Zero-Shot Accuracy: {eval_res.zero_shot_accuracy:.2f}%")
print(f"Memory Accuracy: {eval_res.memory_accuracy:.2f}%")
print(f"Oracle ZS vs Mem: {eval_res.oracle_accuracy:.2f}%")
print(f"Negative Flip Rate (1->0): {neg_rate:.2f}%")

## 6. Cross-Dataset Transfer (Figure 3: AMC23 $\to$ AIME24)

In [ ]:
transfer_pipe = CrossDatasetTransferPipeline(config=config, model=model, tokenizer=tokenizer)

# Use pre-trained pool from AMC23 and evaluate on AIME24
transfer_results = transfer_pipe.evaluate_transfer(aime24_dataset, online_pipeline.memory_pool)
print(f"Zero-shot on AIME24: {transfer_results['zero_shot_accuracy']:.2f}%")
print(f"Transfer ELM on AIME24: {transfer_results['transfer_elm_accuracy']:.2f}%")
print(f"Transfer Gain: {transfer_results['gain']:+.2f}%")

## 7. Run Comparison Baselines (Zero-Shot, ICL, Reasoning Bank, LAG, Verifier@2)

In [ ]:
from elm.pipelines.baselines.zero_shot import ZeroShotCoTPipeline
from elm.pipelines.baselines.icl import InContextLearningPipeline
from elm.pipelines.baselines.reasoning_bank import ReasoningBankPipeline
from elm.pipelines.baselines.lag import LAGPipeline
from elm.pipelines.baselines.verifier_at_2 import VerifierAt2Pipeline

print("Running Baselines...")
zs_acc = ZeroShotCoTPipeline(model, tokenizer).evaluate(amc23_dataset)['accuracy']
icl_acc = InContextLearningPipeline(model, tokenizer).evaluate_stream(amc23_dataset)['accuracy']
rb_acc = ReasoningBankPipeline(model, tokenizer).evaluate_stream(amc23_dataset)['accuracy']
lag_acc = LAGPipeline(model, tokenizer).evaluate_stream(amc23_dataset)['accuracy']
v2_acc = VerifierAt2Pipeline(model, tokenizer).evaluate(amc23_dataset)['accuracy']

print(f"Zero-shot CoT: {zs_acc:.2f}%")
print(f"Online ICL:    {icl_acc:.2f}%")
print(f"Reasoning Bank: {rb_acc:.2f}%")
print(f"LAG:           {lag_acc:.2f}%")
print(f"Verifier@2:    {v2_acc:.2f}%")
print(f"ELM (Ours):    {online_results['accuracy']:.2f}%")

## 8. Generate Paper Plots & Visualizations (Figure 3 & Accuracy Comparison)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Bar chart comparison
methods = ['Zero-Shot', 'ICL', 'Reasoning Bank', 'LAG', 'Verifier@2', 'ELM (Ours)']
scores = [zs_acc, icl_acc, rb_acc, lag_acc, v2_acc, online_results['accuracy']]
colors = ['#888888', '#999999', '#aaaaaa', '#bbbbbb', '#6baed6', '#3182bd']

plt.figure(figsize=(10, 5))
bars = plt.bar(methods, scores, color=colors, edgecolor='black', width=0.6)
plt.ylabel('Pass@1 Accuracy (%)', fontsize=12)
plt.title('Benchmark Comparison on AMC23', fontsize=14, fontweight='bold')
plt.ylim(0, 100)

for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2.0, yval + 1.5, f'{yval:.1f}%', ha='center', va='bottom', fontweight='bold')

plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.savefig('elm_amc23_comparison.png', dpi=300)
plt.show()